# 3D U-Net + FADC (FADC-Full) — MAMA-MIA Breast MRI Segmentation

**Model:** UNet3DFADC — 3D U-Net with Frequency-Adaptive Dilated Convolution at every block (65.2M params)  
**Dataset:** MAMA-MIA (1200 train / 306 val)  
**Baseline to beat:** 3D U-Net Val Dice = 0.6035 (50 epochs, patch 128×128×64)  
**Patch size:** 96×96×48 (reduced from 128×128×64 — FADC's 3 parallel branches require more VRAM)  
**Paper target:** 0.762  
**GPU:** Kaggle T4 x2

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION — edit these before running
# ─────────────────────────────────────────────

DATA_ROOT    = "/kaggle/input/datasets/bharathkumarvemuri/mama-mia-breast-mri-segmentation/mama_mia_kaggle_upload"
OUTPUT_DIR   = "/kaggle/working/outputs/fadc3dunet_full"
CODE_DIR     = "/kaggle/working/FADC-3D"

EPOCHS       = 50
BATCH_SIZE   = 2
NUM_WORKERS  = 4
PATCH_SIZE   = [96, 96, 48]   # reduced from [128,128,64] — FADC 3x parallel branches need more VRAM

# Resume from a previous FADC checkpoint (leave "" for a fresh run)
RESUME_FROM  = ""

# Preprocessed .npz cache — reduces epoch time from ~27min to ~7min
PREPROCESSED_CACHE_DIR = "/kaggle/input/mama-mia-preprocessed-cache"

In [ ]:
# ─────────────────────────────────────────────
# 1. INSTALL DEPENDENCIES
# ─────────────────────────────────────────────
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "monai",
    "--upgrade-strategy", "only-if-needed", "-q"
], check=True)

import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print("Dependencies ready.")

In [ ]:
# ─────────────────────────────────────────────
# 2. CLONE CODE FROM GITHUB
# ─────────────────────────────────────────────
import os

if os.path.exists(CODE_DIR):
    print("Repo already exists — pulling latest...")
    os.system(f"git -C {CODE_DIR} pull")
else:
    os.system(f"git clone https://github.com/Vemuri-BK/FADC-3D.git {CODE_DIR}")
    print("Repo cloned.")

sys.path.insert(0, CODE_DIR)
print(f"Code path: {CODE_DIR}")

In [ ]:
# ─────────────────────────────────────────────
# 3. VERIFY GPU
# ─────────────────────────────────────────────
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU             : {gpu.name}")
    print(f"VRAM            : {gpu.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found — enable GPU in Kaggle settings (Settings → Accelerator → GPU T4 x2)")

In [ ]:
# ─────────────────────────────────────────────
# 4. VERIFY DATASET
# ─────────────────────────────────────────────
from pathlib import Path

data_path  = Path(DATA_ROOT)
images_dir = data_path / "images"
seg_dir    = data_path / "segmentations" / "expert"
csv_path   = data_path / "train_test_splits.csv"

patient_folders = [f for f in images_dir.iterdir() if f.is_dir()]
seg_files       = list(seg_dir.glob("*.nii.gz")) + list(seg_dir.glob("*.nii"))

print(f"Patient folders : {len(patient_folders)}")
print(f"Segmentations   : {len(seg_files)}")
print(f"Split CSV exists: {csv_path.exists()}")

sample = sorted(patient_folders)[0]
print(f"Sample patient  : {sample.name} → {[f.name for f in sample.iterdir()]}")

collections = {}
for f in patient_folders:
    col = f.name.split("_")[0]
    collections[col] = collections.get(col, 0) + 1
for col, count in sorted(collections.items()):
    print(f"  {col}: {count} patients")

In [ ]:
# ─────────────────────────────────────────────
# 5. RUN TRAINING — FADC-Full
# ─────────────────────────────────────────────
import os, subprocess, sys
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_script = os.path.join(CODE_DIR, "training", "train_centralized.py")

cmd = [
    sys.executable, "-u", train_script,
    "--model",       "unet3d_fadc",
    "--data_root",   DATA_ROOT,
    "--output_dir",  OUTPUT_DIR,
    "--epochs",      str(EPOCHS),
    "--batch_size",  str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--patch_size",  str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
]

if RESUME_FROM:
    cmd += ["--resume", RESUME_FROM]

if PREPROCESSED_CACHE_DIR:
    cmd += ["--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR]

print("Command:", " ".join(cmd))
print("=" * 60)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(512)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
process.wait()
print(f"\nExit code: {process.returncode}")

In [ ]:
# ─────────────────────────────────────────────
# 6. PLOT TRAINING CURVES
# ─────────────────────────────────────────────
import json
import matplotlib.pyplot as plt

log_path = os.path.join(OUTPUT_DIR, "train_log.json")

if not os.path.exists(log_path):
    print("No training log found yet.")
else:
    with open(log_path) as f:
        log = json.load(f)

    epochs     = [e["epoch"]    for e in log]
    losses     = [e["loss"]     for e in log]
    val_epochs = [e["epoch"]    for e in log if "val_dice" in e]
    val_dices  = [e["val_dice"] for e in log if "val_dice" in e]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, losses, color="steelblue", linewidth=1.5)
    ax1.set_title("Training Loss", fontsize=13)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.grid(True, alpha=0.3)

    ax2.plot(val_epochs, val_dices, color="darkorange", linewidth=1.5, marker="o", markersize=4)
    ax2.axhline(y=0.6035, color="green",  linestyle="--", linewidth=1.5, label="Our 3D U-Net baseline (0.6035)")
    ax2.axhline(y=0.762,  color="red",    linestyle="--", linewidth=1,   label="MAMA-MIA paper baseline (0.762)")
    ax2.set_title("Validation Dice", fontsize=13)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Dice Score")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    if val_dices:
        best = max(val_dices)
        ax2.set_title(f"Validation Dice  (best: {best:.4f})", fontsize=13)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
    plt.show()
    print(f"Epochs completed : {len(log)}")
    if val_dices:
        print(f"Best Val Dice    : {max(val_dices):.4f}")
        print(f"3D U-Net baseline: 0.6035")
        print(f"Improvement      : {max(val_dices) - 0.6035:+.4f}")

In [ ]:
# ─────────────────────────────────────────────
# 7. SEGMENTATION VISUALIZATION
#    Run after training completes (best_model.pth must exist)
# ─────────────────────────────────────────────
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from matplotlib.patches import Patch

sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc import UNet3DFADC
from data.mama_mia_dataset import build_centralized_loaders
from monai.inferers import sliding_window_inference
from monai.transforms import AsDiscrete

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load best FADC model ───────────────────────────────────────────────────
best_ckpt_path = os.path.join(OUTPUT_DIR, "best_model.pth")
ckpt = torch.load(best_ckpt_path, map_location=device)
cfg  = ckpt["config"]

model = UNet3DFADC(
    in_channels  = cfg["model"]["in_channels"],
    out_channels = cfg["model"]["out_channels"],
    base_filters = cfg["model"]["base_filters"],
).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded FADC best model — Epoch {ckpt['epoch']+1} | Val Dice: {ckpt['best_dice']:.4f}")
print(f"Improvement over 3D U-Net baseline (0.6035): {ckpt['best_dice'] - 0.6035:+.4f}")

# ── Build validation loader ────────────────────────────────────────────────
split_csv = os.path.join(DATA_ROOT, "train_test_splits.csv")
_, val_loader = build_centralized_loaders(
    data_root   = DATA_ROOT,
    split_csv   = split_csv if os.path.exists(split_csv) else None,
    cache_rate  = 0.0,
    num_workers = 0,
    batch_size  = 1,
)

post_pred  = AsDiscrete(argmax=True)
PATCH_SIZE = tuple(cfg["data"]["patch_size"])

# ── Run inference on N_CASES validation volumes ────────────────────────────
N_CASES = 5
results  = []

with torch.no_grad():
    for batch in val_loader:
        if len(results) >= N_CASES:
            break

        images = batch["image"].to(device)
        labels = batch["label"]

        preds    = sliding_window_inference(images, PATCH_SIZE, 1, model, overlap=0.5)
        pred_cls = post_pred(preds[0]).cpu().numpy()
        pred_fg  = (pred_cls[0] == 1).astype(np.float32)
        label_fg = labels[0, 0].numpy()
        image_np = images[0, 0].cpu().numpy()

        tp    = (pred_fg * label_fg).sum()
        denom = pred_fg.sum() + label_fg.sum()
        dice  = float(2 * tp / (denom + 1e-6))

        sums = label_fg.sum(axis=(0, 1))
        z    = int(sums.argmax()) if sums.max() > 0 else image_np.shape[2] // 2

        patient = batch.get("patient_id", ["unknown"])[0] if isinstance(batch, dict) else "?"
        results.append({"image": image_np, "gt": label_fg, "pred": pred_fg,
                        "dice": dice, "z": z, "patient": patient})

print(f"Inference done on {len(results)} validation cases")

# ── Plot: 4 columns per case ───────────────────────────────────────────────
fig, axes = plt.subplots(len(results), 4, figsize=(20, 5 * len(results)))
if len(results) == 1:
    axes = axes[np.newaxis, :]

COL_TITLES = ["MRI Input", "Ground Truth  (green)", "Prediction  (cyan)", "Color Analysis  TP / FP / FN"]

for i, r in enumerate(results):
    img, gt, pred, z = r["image"], r["gt"], r["pred"], r["z"]
    img_norm = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img_s  = img_norm[:, :, z].T
    gt_s   = gt[:, :, z].T
    pred_s = pred[:, :, z].T

    axes[i, 0].imshow(img_s, cmap="gray", origin="lower")
    axes[i, 0].set_ylabel(f"{r['patient']}\nDice = {r['dice']:.3f}", fontsize=9)

    axes[i, 1].imshow(img_s, cmap="gray", origin="lower")
    gt_rgba = np.zeros((*gt_s.shape, 4))
    gt_rgba[gt_s > 0] = [0.0, 1.0, 0.0, 0.55]
    axes[i, 1].imshow(gt_rgba, origin="lower")

    axes[i, 2].imshow(img_s, cmap="gray", origin="lower")
    pred_rgba = np.zeros((*pred_s.shape, 4))
    pred_rgba[pred_s > 0] = [0.0, 0.85, 1.0, 0.55]
    axes[i, 2].imshow(pred_rgba, origin="lower")

    axes[i, 3].imshow(img_s, cmap="gray", origin="lower")
    overlay = np.zeros((*gt_s.shape, 4))
    tp_mask = (gt_s > 0) & (pred_s > 0)
    fp_mask = (gt_s == 0) & (pred_s > 0)
    fn_mask = (gt_s > 0) & (pred_s == 0)
    overlay[tp_mask] = [1.00, 1.00, 0.00, 0.65]
    overlay[fp_mask] = [1.00, 0.20, 0.20, 0.65]
    overlay[fn_mask] = [0.20, 1.00, 0.20, 0.65]
    axes[i, 3].imshow(overlay, origin="lower")

    for ax in axes[i]:
        ax.axis("off")

for j, title in enumerate(COL_TITLES):
    axes[0, j].set_title(title, fontsize=11, fontweight="bold", pad=8)

legend_handles = [
    Patch(facecolor="yellow", label="True Positive  (TP) — correctly detected tumor"),
    Patch(facecolor="red",    label="False Positive (FP) — predicted tumor, actually background"),
    Patch(facecolor="lime",   label="False Negative (FN) — missed tumor"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3,
           fontsize=10, bbox_to_anchor=(0.5, -0.01), framealpha=0.95)

plt.suptitle(
    f"FADC-Full Segmentation Results — Best Model  (Val Dice: {ckpt['best_dice']:.4f}  |  Epoch {ckpt['epoch']+1})",
    fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "segmentation_results.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── Per-case Dice bar chart ────────────────────────────────────────────────
dices    = [r["dice"]    for r in results]
patients = [r["patient"] for r in results]

fig2, ax = plt.subplots(figsize=(max(8, len(results) * 2), 4))
colors = ["#2ecc71" if d >= 0.5 else "#e74c3c" for d in dices]
bars   = ax.bar(patients, dices, color=colors, edgecolor="white", linewidth=0.8)

ax.axhline(0.6035, color="green",      linestyle="--", linewidth=1.5, label="Our 3D U-Net baseline (0.6035)")
ax.axhline(0.762,  color="royalblue",  linestyle="--", linewidth=1.5, label="MAMA-MIA paper baseline (0.762)")
ax.axhline(ckpt["best_dice"], color="darkorange", linestyle="--", linewidth=1.5,
           label=f"FADC best Val Dice ({ckpt['best_dice']:.4f})")
ax.set_ylim(0, 1.0)
ax.set_ylabel("Dice Score", fontsize=11)
ax.set_title("Per-case Dice — Validation Samples (FADC-Full)", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.tick_params(axis="x", rotation=30)

for bar, d in zip(bars, dices):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{d:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "per_case_dice.png"), dpi=150)
plt.show()
print("Visualization complete. Saved to:", OUTPUT_DIR)